In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e4/train.csv
/kaggle/input/competitions/playground-series-s6e4/test.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/O.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/S.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/submission_seed42_c.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/submission_d4_seed2026_a_raw.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/best_single-1.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/I.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/submission_Best_E_fallback.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-

## Setup & Data Loading

In [2]:
import pandas as pd
import numpy as np
 
COMP = '/kaggle/input/competitions/playground-series-s6e4/'
DS07 = '/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/'
 
sub = pd.read_csv(COMP + 'sample_submission.csv')
df2 = pd.read_csv(DS07 + 'R.csv').rename(columns={'Irrigation_Need':'df2'})
df3 = pd.read_csv(DS07 + 'S.csv').rename(columns={'Irrigation_Need':'df3'})
df4 = pd.read_csv(DS07 + 'T.csv')  .rename(columns={'Irrigation_Need':'df4'})
df5 = pd.read_csv(DS07 + 'U.csv')  .rename(columns={'Irrigation_Need':'df5'})
df6 = pd.read_csv(DS07 + 'V.csv')  .rename(columns={'Irrigation_Need':'df6'})
df7 = pd.read_csv(DS07 + 'W.csv')  .rename(columns={'Irrigation_Need':'df7'})
 
# Our best + previous best
our  = pd.read_csv(DS07 + 'submission_d4_s999.csv').rename(columns={'Irrigation_Need':'OUR'})
best = pd.read_csv(DS07 + 'submission_Best_E_fallback.csv').rename(columns={'Irrigation_Need':'BEST'})
dfs = (df2.merge(df3,on='id').merge(df4,on='id').merge(df5,on='id')
          .merge(df6,on='id').merge(df7,on='id').merge(our,on='id').merge(best,on='id'))
 
print(f"Total rows: {len(dfs):,}")

Total rows: 270,000


## Agreement analysis

In [3]:
top4_cols = ['df4','df5','df6','df7']   # the 0.981+ files
all6_cols  = ['df2','df3','df4','df5','df6','df7']
 
for name, cols in [('top4(113-117)', top4_cols), ('all6', all6_cols)]:
    agree = dfs[cols].nunique(axis=1) == 1
    print(f"{name:20s}  agree={agree.sum():,}  disagree={(~agree).sum():,}")

top4(113-117)         agree=269,901  disagree=99
all6                  agree=268,811  disagree=1,189


## Submissions

In [4]:
def voting_v2(row, cols):
    base     = cols[:-1]
    fallback = cols[-1]
    if all(row[c] == row[base[0]] for c in base):
        return row[base[0]]
    return row[fallback]
 
sub1 = sub.copy()
sub1['Irrigation_Need'] = dfs.apply(lambda r: voting_v2(r, all6_cols), axis=1)
sub1.to_csv('submission_BestImprove_exact.csv', index=False)
print(f"\nBestImprove exact dist: {sub1['Irrigation_Need'].value_counts().to_dict()}")
print("Saved: submission_BestImprove_exact.csv")
 

dfs['top4_agree'] = dfs[top4_cols].nunique(axis=1) == 1
 
sub2 = sub.copy()
sub2['Irrigation_Need'] = dfs.apply(
    lambda r: r['df4'] if r['top4_agree'] else r['df7'], axis=1)
sub2.to_csv('submission_Best_top4.csv', index=False)
print(f"Best top4 dist:  {sub2['Irrigation_Need'].value_counts().to_dict()}")
print("Saved: submission_Best_top4.csv")

sub3 = sub.copy()
sub3['Irrigation_Need'] = dfs.apply(
    lambda r: r['df4'] if r['top4_agree'] else r['OUR'], axis=1)
sub3.to_csv('submission_Best_top4_OUR.csv', index=False)
print(f"Best top4+OUR:   {sub3['Irrigation_Need'].value_counts().to_dict()}")
print("Saved: submission_Best_top4_OUR.csv")
 
dfs['all6_agree'] = dfs[all6_cols].nunique(axis=1) == 1
def maj_top4_our(row):
    votes = [row[c] for c in top4_cols] + [row['OUR']]
    return max(set(votes), key=votes.count)
 
sub4 = sub.copy()
sub4['Irrigation_Need'] = dfs.apply(
    lambda r: r['df4'] if r['all6_agree'] else maj_top4_our(r), axis=1)
sub4.to_csv('submission_Best_majority.csv', index=False)
print(f"Best majority:   {sub4['Irrigation_Need'].value_counts().to_dict()}")
print("Saved: submission_Best_majority.csv")


sub5 = sub.copy()
sub5['Irrigation_Need'] = dfs.apply(
    lambda r: r['df4'] if r['all6_agree'] else r['BEST'], axis=1)
sub5.to_csv('submission_Previous_BEST_fallback.csv', index=False)
print(f"Previous+BEST:       {sub5['Irrigation_Need'].value_counts().to_dict()}")
print("Saved: submission_Previous_BEST_fallback.csv")
print("\n" + "="*55)


BestImprove exact dist: {'Low': 159472, 'Medium': 100376, 'High': 10152}
Saved: submission_BestImprove_exact.csv
Best top4 dist:  {'Low': 159476, 'Medium': 100372, 'High': 10152}
Saved: submission_Best_top4.csv
Best top4+OUR:   {'Low': 159482, 'Medium': 100366, 'High': 10152}
Saved: submission_Best_top4_OUR.csv
Best majority:   {'Low': 159480, 'Medium': 100368, 'High': 10152}
Saved: submission_Best_majority.csv
Previous+BEST:       {'Low': 159500, 'Medium': 100307, 'High': 10193}
Saved: submission_Previous_BEST_fallback.csv

